In [16]:
import pandas as pd, numpy as np, re
from pathlib import Path

BASE = Path("/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data")
PAST = BASE / "Combined Past Holdings"
OUT = BASE / "all_holdings_2017_2025.csv"

def to_float(s):
    if s is None:
        return np.nan
    t = str(s).strip()
    if t in {"", "nan", "NA", "N/A", "-", "--", "—"}:
        return np.nan
    t = t.replace("(", "-").replace(")", "")
    t = re.sub(r"[^0-9.\-]", "", t)
    if t.count(".") > 1:
        first, *rest = t.split(".")
        t = first + "." + "".join(rest)
    try:
        return float(t)
    except:
        return np.nan

def load_soi(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    req = {"etf_ticker","etf_name","company_ticker","name_normalized","value"}
    if not req.issubset(df.columns):
        raise ValueError(f"Missing columns in {path.name}: {req - set(df.columns)}")
    df["value"] = df["value"].map(to_float).fillna(0.0)
    if "total_overall_value" in df.columns:
        df["total_overall_value"] = df["total_overall_value"].map(to_float)
    y = re.search(r"soi_(\d{4})_final\.csv$", path.name, flags=re.I)
    if not y:
        raise ValueError(f"Year parse failed for {path.name}")
    year = int(y.group(1))
    gsum = df.groupby("etf_ticker")["value"].transform("sum")
    if "total_overall_value" in df.columns:
        tv = df.groupby("etf_ticker")["total_overall_value"].transform(lambda s: s.dropna().max() if s.notna().any() else np.nan)
        total = np.where(pd.notna(tv) & (tv > 0), tv, gsum)
    else:
        total = gsum
    df = df[["etf_ticker","name_normalized","company_ticker","value"]].copy()
    df["total_value"] = total
    df = df[df["total_value"] > 0]
    df["weight(%)"] = 100.0 * df["value"] / df["total_value"]
    df = df.groupby(["etf_ticker","name_normalized","company_ticker"], as_index=False)["weight(%)"].sum()
    df["date"] = f"{year}-12-31"
    df.rename(columns={"etf_ticker":"ETF_TICKER","name_normalized":"Normalised name","company_ticker":"Company_ticker"}, inplace=True)
    return df[["ETF_TICKER","date","Normalised name","Company_ticker","weight(%)"]]

def load_2025(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    wcol = None
    for cand in ["weight (%)","weight(%)","weight_percent","weight"]:
        if cand in df.columns:
            wcol = cand
            break
    if wcol is None:
        raise ValueError("No weight column found for 2025")
    df["weight(%)"] = df[wcol].apply(lambda x: to_float(str(x).replace("%","")))
    df = df[["etf_ticker","name_normalized","company_ticker","weight(%)"]].copy()
    df = df.groupby(["etf_ticker","name_normalized","company_ticker"], as_index=False)["weight(%)"].sum()
    df["date"] = "2025-12-31"
    df.rename(columns={"etf_ticker":"ETF_TICKER","name_normalized":"Normalised name","company_ticker":"Company_ticker"}, inplace=True)
    return df[["ETF_TICKER","date","Normalised name","Company_ticker","weight(%)"]]

past_files = sorted([p for p in PAST.glob("soi_20*_final.csv") if p.is_file()])
past_frames = [load_soi(p) for p in past_files]
cur_2025 = load_2025(BASE / "holdings_2025_final.csv")

all_holdings = pd.concat(past_frames + [cur_2025], ignore_index=True)
all_holdings["weight(%)"] = all_holdings["weight(%)"].astype(float).round(6)
all_holdings = all_holdings.sort_values(["ETF_TICKER","date","weight(%)"], ascending=[True,True,False]).reset_index(drop=True)
all_holdings.to_csv(OUT, index=False)
print(f"Wrote {OUT}")


KeyError: "['name_normalized'] not in index"

In [33]:
import os
import pandas as pd
import numpy as np

BASE = "/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data"
SRC = os.path.join(BASE, "Controversial_and_Clean_Holdings_final.xlsx")
OUT = os.path.join(BASE, "classification_binary.csv")

def read_all_sheets_xlsx(path):
    xls = pd.ExcelFile(path)
    frames = []
    for s in xls.sheet_names:
        df = pd.read_excel(xls, s)
        if isinstance(df, pd.DataFrame) and not df.empty:
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def pick(df, *cands):
    m = {c.lower().strip(): c for c in df.columns}
    for a in cands:
        if a in m:
            return m[a]
    for a in cands:
        for k in m:
            if a in k:
                return m[k]
    return None

df = read_all_sheets_xlsx(SRC)

c_etf_tic = pick(df, "etf ticker","etf_ticker","ticker")
c_etf_name = pick(df, "etf name","name")
c_screen = pick(df, "screen category","category","screen","tag","label")
c_co_tic = pick(df, "company_ticker","company ticker","ticker")
c_nm = pick(df, "name_normalized","normalised name","normalized name","name")
c_hold_nm = pick(df, "holding name","security","security name","name","name_normalized","normalised name")

df = df.rename(columns={
    c_etf_tic: "ETF Ticker",
    c_etf_name: "ETF Name",
    c_screen: "Screen Category",
    c_co_tic: "company_ticker",
    c_nm: "name_normalized",
    c_hold_nm: "Holding Name"
})[["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized"]]

for col in ["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized"]:
    if col not in df.columns:
        df[col] = np.nan

df["Screen Category"] = df["Screen Category"].astype(str).str.strip()
df["company_ticker"] = df["company_ticker"].astype(str).str.strip()
df["name_normalized"] = df["name_normalized"].astype(str).str.strip()
df["Holding Name"] = df["Holding Name"].astype(str).str.strip()

def flag_contains(s, key):
    s = str(s).lower()
    return int(key in s)

df["clean200"] = df["Screen Category"].str.lower().str.contains("clean200").astype(int)
df["deforestation"] = df["Screen Category"].str.lower().str.contains("deforestation").astype(int)
df["fossil fuel"] = df["Screen Category"].str.lower().str.contains("fossil").astype(int)
df["prison"] = df["Screen Category"].str.lower().str.contains("prison").astype(int)
df["tobacco"] = df["Screen Category"].str.lower().str.contains("tobacco").astype(int)
df["weapons"] = df["Screen Category"].str.lower().str.contains("weapon").astype(int)

grp_keys = ["ETF Ticker","ETF Name","Holding Name","company_ticker","name_normalized"]

agg = (
    df.groupby(grp_keys, as_index=False)
      .agg({
          "Screen Category": lambda x: "; ".join(sorted({str(v).strip() for v in x if pd.notna(v) and str(v).strip() != ""})),
          "clean200": "max",
          "deforestation": "max",
          "fossil fuel": "max",
          "prison": "max",
          "tobacco": "max",
          "weapons": "max"
      })
)

cols = ["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized",
        "clean200","deforestation","fossil fuel","prison","tobacco","weapons"]

agg = agg[cols]
agg.to_csv(OUT, index=False)
print(OUT)


/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/classification_binary.csv


In [39]:
#Analysis 1
import pandas as pd
import numpy as np
import os

BASE_DIR = "/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data"
HOLDINGS_2025 = os.path.join(BASE_DIR, "holdings_2025_final.csv")
CLASS_BIN = os.path.join(BASE_DIR, "classification_binary.csv")
OUT_CONTEXT = os.path.join(BASE_DIR, "context_summary_2025.csv")
OUT_BREAKDOWN = os.path.join(BASE_DIR, "context_breakdown_by_screen.csv")

def pick_col(cols, candidates):
    cl = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cl:
            return cl[cand.lower()]
    for c in cols:
        if c.lower().strip() in [x.lower().strip() for x in candidates]:
            return c
    return None

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).replace(",", "").replace("%", "").strip()
    try:
        return float(s)
    except:
        return np.nan

hold = pd.read_csv(HOLDINGS_2025, dtype=str)
cls = pd.read_csv(CLASS_BIN, dtype=str)

hold_cols = hold.columns.tolist()
cls_cols = cls.columns.tolist()

col_etf = pick_col(hold_cols, ["ETF_Ticker","ETF Ticker","etf_ticker"])
col_name = pick_col(hold_cols, ["name_normalized","Normalised name","Name","Holding Name","security"])
col_tkr = pick_col(hold_cols, ["company_ticker","Company_ticker","Ticker","company_ticker_clean"])
col_wgt = pick_col(hold_cols, ["Weight (%)","weight(%)","weight_percent","weight","Weight"])

clean_col = pick_col(cls_cols, ["clean200"])
ff_col = pick_col(cls_cols, ["fossil fuel","fossil_fuel","fossilfuel"])
weap_col = pick_col(cls_cols, ["weapons"])
tob_col = pick_col(cls_cols, ["tobacco"])
pris_col = pick_col(cls_cols, ["prison"])
defor_col = pick_col(cls_cols, ["deforestation"])
cls_tkr = pick_col(cls_cols, ["company_ticker","Company_ticker"])
cls_name = pick_col(cls_cols, ["name_normalized","Normalised name","name"])

if col_etf is None or col_wgt is None:
    raise ValueError("Missing required columns in holdings file")
if clean_col is None or any(v is None for v in [ff_col,weap_col,tob_col,pris_col,defor_col]) or (cls_tkr is None and cls_name is None):
    raise ValueError("Missing required columns in classification file")
if col_tkr is None and col_name is None:
    raise ValueError("Need either company_ticker or name_normalized in holdings")

hold["_weight"] = hold[col_wgt].apply(to_num).fillna(0.0)

cls_work = cls.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col]:
    cls_work[c] = cls_work[c].astype(str).str.extract(r"(\d+)").fillna("0").astype(int)
cls_work["_any"] = (
    cls_work[ff_col] + cls_work[weap_col] + cls_work[tob_col] + cls_work[pris_col] + cls_work[defor_col]
).clip(upper=1)

if col_tkr is not None and cls_tkr is not None:
    cls_tkr_sub = cls_work[[cls_tkr, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_tkr])
    m_tkr = hold.merge(cls_tkr_sub, left_on=col_tkr, right_on=cls_tkr, how="left")
else:
    m_tkr = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_tkr[c] = np.nan

if col_name is not None and cls_name is not None:
    cls_name_sub = cls_work[[cls_name, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_name]).rename(
        columns={
            clean_col: clean_col+"_n",
            ff_col: ff_col+"_n",
            weap_col: weap_col+"_n",
            tob_col: tob_col+"_n",
            pris_col: pris_col+"_n",
            defor_col: defor_col+"_n",
            "_any": "_any_n"
        }
    )
    m_name = hold.merge(cls_name_sub, left_on=col_name, right_on=cls_name, how="left")
else:
    m_name = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_name[c+"_n"] = np.nan

m = m_tkr.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
    m[c] = m[c].combine_first(m_name[c+"_n"])

m["_clean200"] = m[clean_col].fillna(0).astype(int)
m["_ff"] = m[ff_col].fillna(0).astype(int)
m["_weap"] = m[weap_col].fillna(0).astype(int)
m["_tob"] = m[tob_col].fillna(0).astype(int)
m["_pris"] = m[pris_col].fillna(0).astype(int)
m["_defor"] = m[defor_col].fillna(0).astype(int)
m["_any"] = ((m["_ff"] + m["_weap"] + m["_tob"] + m["_pris"] + m["_defor"]) > 0).astype(int)

m["w_clean"] = m["_weight"] * (m["_clean200"] > 0)
m["w_ff"] = m["_weight"] * (m["_ff"] > 0)
m["w_weap"] = m["_weight"] * (m["_weap"] > 0)
m["w_tob"] = m["_weight"] * (m["_tob"] > 0)
m["w_pris"] = m["_weight"] * (m["_pris"] > 0)
m["w_defor"] = m["_weight"] * (m["_defor"] > 0)
m["w_any"] = m["_weight"] * (m["_any"] > 0)

agg = m.groupby(col_etf, as_index=False).agg(
    total_weight=("_weight","sum"),
    clean_weight=("w_clean","sum"),
    controversial_weight=("w_any","sum")
)
agg["pct_clean"] = 100.0 * agg["clean_weight"] / agg["total_weight"].replace(0, np.nan)
agg["pct_controversial"] = 100.0 * agg["controversial_weight"] / agg["total_weight"].replace(0, np.nan)
agg["pct_other"] = 100.0 - agg[["pct_clean","pct_controversial"]].sum(axis=1)
agg[[col_etf,"pct_clean","pct_controversial","pct_other"]].sort_values(col_etf).to_csv(OUT_CONTEXT, index=False)

screen = m.groupby(col_etf, as_index=False).agg(
    total=("_weight","sum"),
    fossil_fuel=("w_ff","sum"),
    weapons=("w_weap","sum"),
    tobacco=("w_tob","sum"),
    prison=("w_pris","sum"),
    deforestation=("w_defor","sum")
)
for c in ["fossil_fuel","weapons","tobacco","prison","deforestation"]:
    screen[c] = 100.0 * screen[c] / screen["total"].replace(0, np.nan)
screen = screen.melt(id_vars=[col_etf], value_vars=["fossil_fuel","weapons","tobacco","prison","deforestation"], var_name="screen", value_name="pct_weight").dropna()
screen.to_csv(OUT_BREAKDOWN, index=False)

print(f"Wrote: {OUT_CONTEXT}")
print(f"Wrote: {OUT_BREAKDOWN}")


Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/context_summary_2025.csv
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/context_breakdown_by_screen.csv


In [40]:
import pandas as pd
import numpy as np
import os
import re

BASE_DIR = "/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data"
ALL_HOLDINGS = os.path.join(BASE_DIR, "all_holdings_2017_2025.csv")
CLASS_BIN = os.path.join(BASE_DIR, "classification_binary.csv")
OUT_TRENDS = os.path.join(BASE_DIR, "exposure_trends_2017_2025.csv")
OUT_SCREENS = os.path.join(BASE_DIR, "screen_trends_by_category.csv")

def pick_col(cols, candidates):
    cl = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cl:
            return cl[cand.lower()]
    for c in cols:
        if c.lower().strip() in [x.lower().strip() for x in candidates]:
            return c
    return None

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).replace(",", "").replace("%", "").strip()
    try:
        return float(s)
    except:
        return np.nan

def to_year(s):
    try:
        d = pd.to_datetime(s, errors="coerce", dayfirst=True)
        if pd.notna(d):
            return int(d.year)
    except:
        pass
    m = re.search(r"(20\d{2}|19\d{2})", str(s))
    return int(m.group(1)) if m else np.nan

hold = pd.read_csv(ALL_HOLDINGS, dtype=str)
cls = pd.read_csv(CLASS_BIN, dtype=str)

hcols = hold.columns.tolist()
ccols = cls.columns.tolist()

col_etf = pick_col(hcols, ["ETF_TICKER","ETF Ticker","etf_ticker"])
col_date = pick_col(hcols, ["date","Date"])
col_name = pick_col(hcols, ["Normalised name","name_normalized","Name","Holding Name","security"])
col_tkr = pick_col(hcols, ["Company_ticker","company_ticker","Ticker","company_ticker_clean"])
col_wgt = pick_col(hcols, ["weight(%)","Weight (%)","weight_percent","weight","Weight"])

clean_col = pick_col(ccols, ["clean200"])
ff_col = pick_col(ccols, ["fossil fuel","fossil_fuel","fossilfuel"])
weap_col = pick_col(ccols, ["weapons"])
tob_col = pick_col(ccols, ["tobacco"])
pris_col = pick_col(ccols, ["prison"])
defor_col = pick_col(ccols, ["deforestation"])
cls_tkr = pick_col(ccols, ["company_ticker","Company_ticker"])
cls_name = pick_col(ccols, ["name_normalized","Normalised name","name"])

if any(x is None for x in [col_etf, col_date, col_wgt]) or (col_tkr is None and col_name is None):
    raise ValueError("Missing required columns in all_holdings_2017_2025.csv")
if clean_col is None or any(v is None for v in [ff_col,weap_col,tob_col,pris_col,defor_col]) or (cls_tkr is None and cls_name is None):
    raise ValueError("Missing required columns in classification_binary.csv")

hold["_weight"] = hold[col_wgt].apply(to_num).fillna(0.0)
hold["_year"] = hold[col_date].apply(to_year)
hold = hold[pd.notna(hold["_year"])]

cls_work = cls.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col]:
    cls_work[c] = cls_work[c].astype(str).str.extract(r"(\d+)").fillna("0").astype(int)
cls_work["_any"] = (
    cls_work[ff_col] + cls_work[weap_col] + cls_work[tob_col] + cls_work[pris_col] + cls_work[defor_col]
).clip(upper=1)

if col_tkr is not None and cls_tkr is not None:
    sub_tkr = cls_work[[cls_tkr, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_tkr])
    m_tkr = hold.merge(sub_tkr, left_on=col_tkr, right_on=cls_tkr, how="left")
else:
    m_tkr = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_tkr[c] = np.nan

if col_name is not None and cls_name is not None:
    sub_name = cls_work[[cls_name, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_name]).rename(
        columns={
            clean_col: clean_col+"_n",
            ff_col: ff_col+"_n",
            weap_col: weap_col+"_n",
            tob_col: tob_col+"_n",
            pris_col: pris_col+"_n",
            defor_col: defor_col+"_n",
            "_any": "_any_n"
        }
    )
    m_name = hold.merge(sub_name, left_on=col_name, right_on=cls_name, how="left")
else:
    m_name = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_name[c+"_n"] = np.nan

m = m_tkr.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
    m[c] = m[c].combine_first(m_name[c+"_n"])

m["_clean200"] = m[clean_col].fillna(0).astype(int)
m["_ff"] = m[ff_col].fillna(0).astype(int)
m["_weap"] = m[weap_col].fillna(0).astype(int)
m["_tob"] = m[tob_col].fillna(0).astype(int)
m["_pris"] = m[pris_col].fillna(0).astype(int)
m["_defor"] = m[defor_col].fillna(0).astype(int)
m["_any"] = ((m["_ff"] + m["_weap"] + m["_tob"] + m["_pris"] + m["_defor"]) > 0).astype(int)

m["w_clean"] = m["_weight"] * (m["_clean200"] > 0)
m["w_ff"] = m["_weight"] * (m["_ff"] > 0)
m["w_weap"] = m["_weight"] * (m["_weap"] > 0)
m["w_tob"] = m["_weight"] * (m["_tob"] > 0)
m["w_pris"] = m["_weight"] * (m["_pris"] > 0)
m["w_defor"] = m["_weight"] * (m["_defor"] > 0)
m["w_any"] = m["_weight"] * (m["_any"] > 0)

agg = (
    m.groupby([col_etf, "_year"], as_index=False)
     .agg(total_weight=("_weight","sum"),
          clean_weight=("w_clean","sum"),
          controversial_weight=("w_any","sum"))
)
agg["pct_clean"] = 100.0 * agg["clean_weight"] / agg["total_weight"].replace(0, np.nan)
agg["pct_controversial"] = 100.0 * agg["controversial_weight"] / agg["total_weight"].replace(0, np.nan)
agg["pct_other"] = 100.0 - agg[["pct_clean","pct_controversial"]].sum(axis=1)
exposure_trends = agg[[col_etf, "_year", "pct_clean", "pct_controversial", "pct_other"]].rename(columns={"_year":"year"})
exposure_trends = exposure_trends.sort_values([col_etf, "year"]).reset_index(drop=True)
exposure_trends.to_csv(OUT_TRENDS, index=False)

scr = (
    m.groupby([col_etf, "_year"], as_index=False)
     .agg(total=("_weight","sum"),
          fossil_fuel=("w_ff","sum"),
          weapons=("w_weap","sum"),
          tobacco=("w_tob","sum"),
          prison=("w_pris","sum"),
          deforestation=("w_defor","sum"))
)
for c in ["fossil_fuel","weapons","tobacco","prison","deforestation"]:
    scr[c] = 100.0 * scr[c] / scr["total"].replace(0, np.nan)
scr_long = scr.melt(id_vars=[col_etf, "_year"], value_vars=["fossil_fuel","weapons","tobacco","prison","deforestation"], var_name="screen", value_name="pct_weight").rename(columns={"_year":"year"})
scr_long = scr_long.sort_values([col_etf, "year", "screen"]).reset_index(drop=True)
scr_long.to_csv(OUT_SCREENS, index=False)

print(f"Wrote: {OUT_TRENDS}")
print(f"Wrote: {OUT_SCREENS}")


/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_43736/1683396990.py:35: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d = pd.to_datetime(s, errors="coerce", dayfirst=True)


Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/exposure_trends_2017_2025.csv
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/screen_trends_by_category.csv


In [46]:
import pandas as pd
import numpy as np
import os
import glob
import re

BASE_DIR = "/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data"
PAST_DIR = os.path.join(BASE_DIR, "Combined Past Holdings")
HOLDINGS_2025 = os.path.join(BASE_DIR, "holdings_2025_final.csv")
EXPOSURE_TRENDS = os.path.join(BASE_DIR, "exposure_trends_2017_2025.csv")
SCREEN_TRENDS = os.path.join(BASE_DIR, "screen_trends_by_category.csv")

OUT_AUM = os.path.join(BASE_DIR, "etf_aum_by_year.csv")
OUT_AGG_EXP = os.path.join(BASE_DIR, "aggregate_exposure_trends.csv")
OUT_AGG_SCR = os.path.join(BASE_DIR, "aggregate_screen_trends.csv")
OUT_DISP = os.path.join(BASE_DIR, "exposure_dispersion_stats.csv")

YEAR_MIN, YEAR_MAX = 2017, 2025

def pick_col(cols, candidates):
    cl = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cl:
            return cl[cand.lower()]
    for c in cols:
        if c.lower().strip() in [x.lower().strip() for x in candidates]:
            return c
    return None

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x)
    s = s.replace(",","").replace("%","").replace("₹","").replace("$","").replace("€","").replace("£","").strip()
    try:
        return float(s)
    except:
        return np.nan

def to_year(s):
    try:
        d = pd.to_datetime(s, errors="coerce", dayfirst=True)
        if pd.notna(d):
            y = int(d.year)
            if YEAR_MIN-10 <= y <= YEAR_MAX:
                return y
    except:
        pass
    m = re.search(r"(20\d{2}|19\d{2})", str(s))
    if m:
        y = int(m.group(1))
        if YEAR_MIN-10 <= y <= YEAR_MAX:
            return y
    return np.nan

def extract_aum_from_soi(path, year_hint=None):
    df = pd.read_csv(path, dtype=str)
    etf_col = pick_col(df.columns, ["etf_ticker","ETF_Ticker","ETF Ticker"])
    date_col = pick_col(df.columns, ["date","Date"])
    val_col = pick_col(df.columns, ["value","holding_value","market_value","Value","Market Value"])
    tot_val_col = pick_col(df.columns, ["total_value","Total Value","fund_total_value"])
    tot_overall_col = pick_col(df.columns, ["total_overall_value","Total Overall Value","fund_total_overall_value","fund_total"])
    if etf_col is None:
        return pd.DataFrame(columns=["ETF_TICKER","year","AUM"])
    df["_etf"] = df[etf_col].astype(str)
    if date_col is not None:
        df["_year"] = df[date_col].apply(to_year)
    else:
        df["_year"] = year_hint
    for c in [val_col, tot_val_col, tot_overall_col]:
        if c is not None:
            df[c] = df[c].apply(to_num)
    parts = []
    for (e, y), g in df.groupby(["_etf","_year"]):
        if pd.isna(y):
            continue
        aum = np.nan
        if tot_overall_col is not None and g[tot_overall_col].notna().any():
            aum = g[tot_overall_col].dropna().iloc[-1]
        if (np.isnan(aum) or aum == 0) and tot_val_col is not None and g[tot_val_col].notna().any():
            aum = g[tot_val_col].dropna().iloc[-1]
        if (np.isnan(aum) or aum == 0) and val_col is not None:
            aum = g[val_col].sum()
        parts.append({"ETF_TICKER": e, "year": int(y), "AUM": float(aum if pd.notna(aum) else 0.0)})
    out = pd.DataFrame(parts)
    if not out.empty:
        out = out[(out["year"] >= YEAR_MIN) & (out["year"] <= YEAR_MAX)]
    return out

def extract_aum_2025(path):
    df = pd.read_csv(path, dtype=str)
    etf_col = pick_col(df.columns, ["ETF_Ticker","ETF Ticker","etf_ticker"])
    tot_overall_col = pick_col(df.columns, ["total_overall_value","Total Overall Value"])
    tot_val_col = pick_col(df.columns, ["total_value","Total Value"])
    mval_col = pick_col(df.columns, ["Market Value","market_value","value"])
    if etf_col is None:
        return pd.DataFrame(columns=["ETF_TICKER","year","AUM"])
    df["_etf"] = df[etf_col].astype(str)
    for c in [tot_overall_col, tot_val_col, mval_col]:
        if c is not None:
            df[c] = df[c].apply(to_num)
    parts = []
    for e, g in df.groupby("_etf"):
        aum = np.nan
        if tot_overall_col is not None and g[tot_overall_col].notna().any():
            aum = g[tot_overall_col].dropna().iloc[-1]
        if (np.isnan(aum) or aum == 0) and tot_val_col is not None and g[tot_val_col].notna().any():
            aum = g[tot_val_col].dropna().iloc[-1]
        if (np.isnan(aum) or aum == 0) and mval_col is not None:
            aum = g[mval_col].sum()
        parts.append({"ETF_TICKER": e, "year": 2025, "AUM": float(aum if pd.notna(aum) else 0.0)})
    out = pd.DataFrame(parts)
    out = out[(out["year"] >= YEAR_MIN) & (out["year"] <= YEAR_MAX)]
    return out

soi_paths = sorted(glob.glob(os.path.join(PAST_DIR, "soi_20*_final.csv")))
year_from_name = []
for p in soi_paths:
    b = os.path.basename(p)
    yr = None
    for tok in re.findall(r"\d{4}", b):
        if tok.isdigit():
            yr = int(tok)
    year_from_name.append((p, yr))

aum_frames = []
for p, yr in year_from_name:
    aum_frames.append(extract_aum_from_soi(p, yr))
aum_frames.append(extract_aum_2025(HOLDINGS_2025))
aum_by_year = pd.concat(aum_frames, ignore_index=True) if len(aum_frames) else pd.DataFrame(columns=["ETF_TICKER","year","AUM"])
aum_by_year = aum_by_year.dropna(subset=["year"])
aum_by_year["year"] = aum_by_year["year"].astype(int)
aum_by_year = aum_by_year[(aum_by_year["year"] >= YEAR_MIN) & (aum_by_year["year"] <= YEAR_MAX)]
aum_by_year = aum_by_year.groupby(["ETF_TICKER","year"], as_index=False)["AUM"].sum()
aum_by_year.to_csv(OUT_AUM, index=False)

exp = pd.read_csv(EXPOSURE_TRENDS)
col_etf = pick_col(exp.columns, ["ETF_TICKER","ETF Ticker","etf_ticker"])
col_year = pick_col(exp.columns, ["year","Year"])
exp = exp.rename(columns={col_etf:"ETF_TICKER", col_year:"year"})
for c in ["pct_clean","pct_controversial","pct_other"]:
    exp[c] = exp[c].apply(to_num)
exp = exp[(exp["year"] >= YEAR_MIN) & (exp["year"] <= YEAR_MAX)]

scr = pd.read_csv(SCREEN_TRENDS)
col_etf2 = pick_col(scr.columns, ["ETF_TICKER","ETF Ticker","etf_ticker"])
col_year2 = pick_col(scr.columns, ["year","Year"])
scr = scr.rename(columns={col_etf2:"ETF_TICKER", col_year2:"year"})
scr["pct_weight"] = scr["pct_weight"].apply(to_num)
scr = scr[(scr["year"] >= YEAR_MIN) & (scr["year"] <= YEAR_MAX)]

exp_w = exp.merge(aum_by_year, on=["ETF_TICKER","year"], how="left")
exp_w["AUM"] = exp_w["AUM"].fillna(0.0)
agg_exp = exp_w.groupby("year", as_index=False).apply(
    lambda g: pd.Series({
        "total_aum": g["AUM"].sum(),
        "pct_clean_w": (g["pct_clean"]*g["AUM"]).sum()/g["AUM"].sum() if g["AUM"].sum()>0 else np.nan,
        "pct_controversial_w": (g["pct_controversial"]*g["AUM"]).sum()/g["AUM"].sum() if g["AUM"].sum()>0 else np.nan
    })
).reset_index(drop=True)
agg_exp["pct_other_w"] = 100.0 - (agg_exp["pct_clean_w"] + agg_exp["pct_controversial_w"])
agg_exp = agg_exp[["year","pct_clean_w","pct_controversial_w","pct_other_w","total_aum"]].sort_values("year")
agg_exp.to_csv(OUT_AGG_EXP, index=False)

scr_w = scr.merge(aum_by_year, on=["ETF_TICKER","year"], how="left")
scr_w["AUM"] = scr_w["AUM"].fillna(0.0)
agg_scr = scr_w.groupby(["year","screen"], as_index=False).apply(
    lambda g: pd.Series({
        "total_aum": g["AUM"].sum(),
        "pct_weight_w": (g["pct_weight"]*g["AUM"]).sum()/g["AUM"].sum() if g["AUM"].sum()>0 else np.nan
    })
).reset_index(drop=True)
agg_scr = agg_scr.sort_values(["year","screen"])
agg_scr.to_csv(OUT_AGG_SCR, index=False)

disp = exp.groupby("year").agg(
    pct_clean_min=("pct_clean","min"),
    pct_clean_max=("pct_clean","max"),
    pct_clean_mean=("pct_clean","mean"),
    pct_clean_median=("pct_clean","median"),
    pct_clean_std=("pct_clean","std"),
    pct_cont_min=("pct_controversial","min"),
    pct_cont_max=("pct_controversial","max"),
    pct_cont_mean=("pct_controversial","mean"),
    pct_cont_median=("pct_controversial","median"),
    pct_cont_std=("pct_controversial","std"),
    pct_other_min=("pct_other","min"),
    pct_other_max=("pct_other","max"),
    pct_other_mean=("pct_other","mean"),
    pct_other_median=("pct_other","median"),
    pct_other_std=("pct_other","std"),
    etf_count=("pct_clean","count")
).reset_index().sort_values("year")
disp.to_csv(OUT_DISP, index=False)

print(f"Wrote: {OUT_AUM}")
print(f"Wrote: {OUT_AGG_EXP}")
print(f"Wrote: {OUT_AGG_SCR}")
print(f"Wrote: {OUT_DISP}")


Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/etf_aum_by_year.csv
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/aggregate_exposure_trends.csv
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/aggregate_screen_trends.csv
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data/exposure_dispersion_stats.csv


/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_43736/2829156407.py:156: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg_exp = exp_w.groupby("year", as_index=False).apply(
/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_43736/2829156407.py:169: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg_scr = scr_w.groupby(["year","screen"], as_index=False).apply(


In [ ]:
import re, pandas as pd, numpy as np, time, sys, os
from pathlib import Path
from datetime import datetime
import yfinance as yf

BASE = Path("/Users/nityaarya/Downloads/blackrock-esg-etf-study/blackrock-esg-etf-study/Data/Final Data")
HOLD_FILE = BASE / "holdings_2025_final.csv"
ALIASES_FILE = BASE / "ticker_aliases.csv"
OUT_PRICES = BASE / "security_prices.csv"
OUT_FAIL = BASE / "failed_tickers.csv"
OUT_LOG = BASE / "download_log.txt"

RETRIES = 4
BASE_SLEEP = 1.0
PAUSE_EVERY = 50
PAUSE_SECS = 1.0
TIMEOUT_MSG = "no_data_or_timeout"

def log(msg, end="\n"):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, end=end, flush=True)
    with open(OUT_LOG, "a") as f:
        f.write(line + "\n")

def pick(df, names):
    m = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in m: return m[n.lower()]
    return None

def parse_dates_any(s):
    d = pd.to_datetime(s, errors="coerce", dayfirst=True)
    if d.notna().any(): return d
    d = pd.to_datetime(s, errors="coerce")
    return d

def load_holdings_universe():
    if not HOLD_FILE.exists():
        raise FileNotFoundError(f"Missing {HOLD_FILE}")
    df = pd.read_csv(HOLD_FILE)
    df.columns = [c.strip() for c in df.columns]
    tcol = pick(df, ["Company_ticker","company_ticker","Ticker","ticker"])
    dcol = pick(df, ["date","Date","DATE"])
    if tcol is None:
        raise ValueError("company_ticker column not found in holdings_2025_final.csv")
    df["company_ticker"] = df[tcol].astype(str).str.strip()
    df["date"] = parse_dates_any(df[dcol]) if dcol is not None else pd.NaT
    df = df.dropna(subset=["company_ticker"])
    df = df[~df["company_ticker"].str.startswith("ZZ_")]
    df = df[~df["company_ticker"].str.contains(r"(?:^|[^A-Z])(CASH|USD|EUR|GBP)(?:$|[^A-Z])", case=False, na=False)]
    return df[["company_ticker","date"]]

def date_range(universe):
    dmin = universe["date"].min()
    dmax = universe["date"].max()
    if pd.isna(dmin): dmin = pd.Timestamp("2016-01-01")
    if pd.isna(dmax): dmax = pd.Timestamp.today()
    return (dmin - pd.Timedelta(days=10)).normalize(), (dmax + pd.Timedelta(days=10)).normalize()

def heuristic_alias(t):
    u = t.strip().upper()
    if u.startswith("ZZ_"): return None
    if u == "BT.A": return "BT-A.L"
    if re.fullmatch(r"\d{4}", u): return u + ".T"
    if re.fullmatch(r"\d{1,4}", u):
        s = u.zfill(4)
        return s + ".HK"
    if u in {"ULVR","LLOY","BARC","SVT","EXPN","KGF","ITRK","SDR","CRDA","BNZL","WTB","ANTO","WPP","PSON","ABF"}: return u + ".L"
    if u in {"RR.","AV.","NG.","UU.","BA."}: return u.replace(".","") + ".L"
    if u in {"PHIA","HEIA","KPN","WKL"}: return u + ".AS"
    if u in {"RNO","HO"}: return u + ".PA"
    if u in {"DB1","CBK","G1A"}: return u + ".DE"
    if u in {"UBSG","ZURN"}: return u + ".SW"
    if u in {"NAB","WBC","BXB","QBE"}: return u + ".AX"
    if u in {"D05","U11","H78","O39","Z74"}: return u + ".SI"
    if re.fullmatch(r"[A-Z]{1,4}\d{1,2}", u) or u.endswith("3"): return u + ".SA"
    if u in {"NPN"}: return u + ".JO"
    if u in {"GALP"}: return u + ".LS"
    if u in {"RELIANCE","TCS","HINDUNILVR","ASIANPAINT","HCLTECH","BHARTIARTL","ICICIBANK","AXISBANK","WIPRO","NESTLEIND","INFY","ITC"}: return u + ".NS"
    if u == "GIB.A": return "GIB-A.TO"
    if u == "BRKB": return "BRK-B"
    if re.fullmatch(r"[A-Z]{1,5}(\-[A-Z])?", u): return u
    return None

def build_or_update_aliases():
    uni = load_holdings_universe()
    cur_alias = pd.DataFrame(columns=["company_ticker","yahoo_ticker"])
    if ALIASES_FILE.exists():
        cur_alias = pd.read_csv(ALIASES_FILE)
        cur_alias.columns = [c.strip() for c in cur_alias.columns]
        if "company_ticker" not in cur_alias.columns or "yahoo_ticker" not in cur_alias.columns:
            cur_alias = pd.DataFrame(columns=["company_ticker","yahoo_ticker"])
        else:
            cur_alias["company_ticker"] = cur_alias["company_ticker"].astype(str).str.strip()
            cur_alias["yahoo_ticker"] = cur_alias["yahoo_ticker"].astype(str).str.strip()
    uniq_ticks = uni["company_ticker"].dropna().unique().tolist()
    heur = []
    for t in uniq_ticks:
        if cur_alias.shape[0] and (cur_alias["company_ticker"] == t).any():
            continue
        m = heuristic_alias(t)
        if m: heur.append((t, m, "heuristic"))
    add_df = pd.DataFrame(heur, columns=["company_ticker","yahoo_ticker","source"]) if heur else pd.DataFrame(columns=["company_ticker","yahoo_ticker","source"])
    merged = cur_alias.merge(add_df[["company_ticker","yahoo_ticker"]], on="company_ticker", how="outer", suffixes=("","_heur"))
    merged["yahoo_ticker"] = np.where(merged["yahoo_ticker"].notna(), merged["yahoo_ticker"], merged["yahoo_ticker_heur"])
    merged = merged.drop(columns=[c for c in merged.columns if c.endswith("_heur")]).dropna(subset=["company_ticker","yahoo_ticker"])
    merged = merged.drop_duplicates(subset=["company_ticker"]).sort_values("company_ticker")
    merged.to_csv(ALIASES_FILE, index=False)
    return merged, uni

def already_done():
    if not OUT_PRICES.exists(): return set()
    try:
        done = pd.read_csv(OUT_PRICES, usecols=["company_ticker"])
        return set(done["company_ticker"].unique().tolist())
    except Exception:
        return set()

def already_failed():
    if not OUT_FAIL.exists(): return {}
    try:
        f = pd.read_csv(OUT_FAIL)
        if "company_ticker" in f.columns and "yahoo_ticker" in f.columns:
            return {(r["company_ticker"], r["yahoo_ticker"]): r["reason"] for _, r in f.iterrows()}
        return {}
    except Exception:
        return {}

def append_prices(df):
    mode = "a" if OUT_PRICES.exists() else "w"
    header = not OUT_PRICES.exists()
    df.to_csv(OUT_PRICES, mode=mode, header=header, index=False)

def append_fail(rows):
    if not rows: return
    fdf = pd.DataFrame(rows, columns=["company_ticker","yahoo_ticker","reason"])
    mode = "a" if OUT_FAIL.exists() else "w"
    header = not OUT_FAIL.exists()
    fdf.to_csv(OUT_FAIL, mode=mode, header=header, index=False)

def dl_one(sym, start, end):
    last_err = None
    for k in range(RETRIES):
        try:
            t = yf.Ticker(sym)
            df = t.history(start=start, end=end, auto_adjust=True, actions=False, interval="1d")
            if isinstance(df, pd.DataFrame) and not df.empty:
                s = df["Close"].rename(sym).to_frame()
                s.index = pd.to_datetime(s.index)
                return s
            time.sleep(BASE_SLEEP * (k+1))
        except Exception as e:
            last_err = str(e)
            time.sleep(BASE_SLEEP * (k+1))
    return last_err if last_err else TIMEOUT_MSG

def main():
    log("building/merging aliases from holdings_2025_final.csv")
    aliases, uni = build_or_update_aliases()
    dmin, dmax = date_range(uni)

    ymap = aliases.copy()
    ymap["company_ticker"] = ymap["company_ticker"].astype(str).str.strip()
    ymap["yahoo_ticker"] = ymap["yahoo_ticker"].astype(str).str.strip()
    ymap = ymap[ymap["yahoo_ticker"].str.len() > 0]

    if ymap.empty:
        log("no valid tickers to download")
        pd.DataFrame(columns=["company_ticker","date","price"]).to_csv(OUT_PRICES, index=False)
        pd.DataFrame([{"company_ticker":"", "yahoo_ticker":"", "reason":"no_valid_tickers"}]).to_csv(OUT_FAIL, index=False)
        return

    done = already_done()
    prev_fail = already_failed()

    todo = ymap[~ymap["company_ticker"].isin(done)].copy()
    total = len(todo)
    if total == 0:
        log("nothing to do; all tickers already downloaded")
        return

    log(f"date_range={dmin.date()} \u2192 {dmax.date()}")
    log(f"tickers_total={total} tickers_skipped_done={len(done)} prev_fail_records={len(prev_fail)}")

    successes = 0
    fails = 0
    fail_buf = []

    cmap = dict(zip(todo["yahoo_ticker"], todo["company_ticker"]))
    reverse_alias = dict(zip(aliases["company_ticker"], aliases["yahoo_ticker"]))
    syms = todo["yahoo_ticker"].tolist()

    for i, sym in enumerate(syms, 1):
        ct = cmap.get(sym, "")
        alias_now = reverse_alias.get(ct, sym)
        prev = prev_fail.get((ct, sym))
        if prev and alias_now == sym:
            log(f"{i}/{total} skip_failed prev={prev} {ct} <- {sym}")
            fails += 1
            continue

        start_t = time.time()
        log(f"{i}/{total} start {ct} <- {sym}")
        res = dl_one(sym, dmin, dmax)

        if isinstance(res, pd.DataFrame):
            long = res.reset_index().rename(columns={"Date":"date", sym:"price"}).dropna(subset=["price"])
            if not long.empty:
                long["company_ticker"] = ct
                append_prices(long[["company_ticker","date","price"]].sort_values(["company_ticker","date"]))
                successes += 1
                dur = time.time() - start_t
                log(f"{i}/{total} ok rows={len(long)} dur={dur:.2f}s done={successes} fail={fails}")
            else:
                fails += 1
                fail_buf.append((ct, sym, "empty"))
                log(f"{i}/{total} fail empty done={successes} fail={fails}")
        else:
            fails += 1
            fail_buf.append((ct, sym, res))
            log(f"{i}/{total} fail {res} done={successes} fail={fails}")

        if i % 10 == 0 and fail_buf:
            append_fail(fail_buf)
            fail_buf = []

        if i % PAUSE_EVERY == 0:
            time.sleep(PAUSE_SECS)

    if fail_buf:
        append_fail(fail_buf)

    log(f"finished tickers_total={total} success={successes} failed={fails}")
    log(f"outputs: {OUT_PRICES.name}, {OUT_FAIL.name}, {OUT_LOG.name}, {ALIASES_FILE.name}")

if __name__ == "__main__":
    main()


[2025-09-23 15:25:21] building/merging aliases from holdings_2025_final.csv


/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_56080/3632427269.py:33: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_56080/3632427269.py:51: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["company_ticker"].str.contains(r"(?:^|[^A-Z])(CASH|USD|EUR|GBP)(?:$|[^A-Z])", case=False, na=False)]


[2025-09-23 15:25:22] date_range=2017-12-21 → 2026-01-10
[2025-09-23 15:25:22] tickers_total=3348 tickers_skipped_done=0 prev_fail_records=0
[2025-09-23 15:25:22] 1/3348 start 1 <- 0001.HK
[2025-09-23 15:25:22] 1/3348 ok rows=1907 dur=0.61s done=1 fail=0
[2025-09-23 15:25:22] 2/3348 start 100 <- 0100.HK


$0100.HK: possibly delisted; no timezone found
$0100.HK: possibly delisted; no timezone found
$0100.HK: possibly delisted; no timezone found
$0100.HK: possibly delisted; no timezone found


[2025-09-23 15:25:33] 2/3348 fail no_data_or_timeout done=1 fail=1
[2025-09-23 15:25:33] 3/3348 start 1010 <- 1010.T


...

[2025-09-23 17:32:15] 773/3348 fail no_data_or_timeout done=607 fail=166
[2025-09-23 17:32:15] 774/3348 start BEN <- BEN
[2025-09-23 17:32:16] 774/3348 ok rows=1947 dur=0.43s done=608 fail=166
[2025-09-23 17:32:16] 775/3348 start BEPC <- BEPC
[2025-09-23 17:32:16] 775/3348 ok rows=1297 dur=0.17s done=609 fail=166
[2025-09-23 17:32:16] 776/3348 start BESI <- BESI


$BESI: possibly delisted; no price data found  (1d 2017-12-21 00:00:00 -> 2026-01-10 00:00:00)
$BESI: possibly delisted; no price data found  (1d 2017-12-21 00:00:00 -> 2026-01-10 00:00:00)
$BESI: possibly delisted; no price data found  (1d 2017-12-21 00:00:00 -> 2026-01-10 00:00:00)
$BESI: possibly delisted; no price data found  (1d 2017-12-21 00:00:00 -> 2026-01-10 00:00:00)


[2025-09-23 17:32:26] 776/3348 fail no_data_or_timeout done=609 fail=167
[2025-09-23 17:32:26] 777/3348 start BF-B <- BF-B
[2025-09-23 17:32:27] 777/3348 ok rows=1947 dur=0.40s done=610 fail=167
[2025-09-23 17:32:27] 778/3348 start BFAM <- BFAM
[2025-09-23 17:32:27] 778/3348 ok rows=1947 dur=0.19s done=611 fail=167
[2025-09-23 17:32:27] 779/3348 start BFB <- BFB


$BFB: possibly delisted; no timezone found
$BFB: possibly delisted; no timezone found
$BFB: possibly delisted; no timezone found
$BFB: possibly delisted; no timezone found


[2025-09-23 17:32:37] 779/3348 fail no_data_or_timeout done=611 fail=168
[2025-09-23 17:32:37] 780/3348 start BFFAF <- BFFAF
[2025-09-23 17:32:38] 780/3348 ok rows=1947 dur=0.29s done=612 fail=168
[2025-09-23 17:32:38] 781/3348 start BFH <- BFH
[2025-09-23 17:32:38] 781/3348 ok rows=1947 dur=0.31s done=613 fail=168
[2025-09-23 17:32:38] 782/3348 start BG <- BG
[2025-09-23 17:32:38] 782/3348 ok rows=1947 dur=0.23s done=614 fail=168
[2025-09-23 17:32:38] 783/3348 start BGC <- BGC
[2025-09-23 17:32:38] 783/3348 ok rows=1947 dur=0.21s done=615 fail=168
[2025-09-23 17:32:38] 784/3348 start BGS <- BGS
[2025-09-23 17:32:39] 784/3348 ok rows=1947 dur=0.20s done=616 fail=168
[2025-09-23 17:32:39] 785/3348 start BHARTIARTL <- BHARTIARTL.NS
[2025-09-23 17:32:39] 785/3348 ok rows=1916 dur=0.34s done=617 fail=168
[2025-09-23 17:32:39] 786/3348 start BHC <- BHC
[2025-09-23 17:32:39] 786/3348 ok rows=1947 dur=0.17s done=618 fail=168
[2025-09-23 17:32:39] 787/3348 start BHE <- BHE
[2025-09-23 17:32:39